# 文本摘要示例

## Step1 导入包

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

## Step2 加载数据集

In [ ]:
ds = Dataset.load_from_disk("./nlpcc_2017/")
ds

In [ ]:
ds = ds.train_test_split(100, seed = 42)
ds

In [ ]:
ds["train"][0]

## Step3 数据处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("THUDM/glm-large-chinese", trust_remote_code=True)
tokenizer

In [ ]:
?tokenizer
tokenizer.mask_token


In [ ]:
help(tokenizer)

In [ ]:
def process_func(examples):
    contents = ["摘要生成：\n" + example_content + tokenizer.mask_token for example_content in examples["content"]]
    inputs = tokenizer(
        contents,
        max_length=384,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    inputs = tokenizer.build_inputs_for_generation(inputs, targets=examples["title"], padding=True, max_gen_length=64)
    return inputs

In [ ]:
tokenized_ds = ds.map(process_func, batched=True,remove_columns=ds["train"].column_names)


In [ ]:
tokenized_ds

In [ ]:
print(tokenized_ds["train"][0]["input_ids"])
print(tokenizer.decode(tokenized_ds["train"][0]["input_ids"]))

In [ ]:
print(tokenized_ds["train"][0]["position_ids"])

## Step4 创建模型

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("THUDM/glm-large-chinese", trust_remote_code=True)

## Step6 创建训练参数

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary_glm",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    logging_steps=8,
    num_train_epochs=1
)

## Step7 创建训练器

In [ ]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_ds["train"],
)

## Step8 训练模型

In [ ]:
trainer.train()

## Step9 模型推理

In [ ]:
input_text = ds["test"][-1]["content"]
inputs = tokenizer(
    "摘要生成: \n" + input_text + tokenizer.mask_token, return_tensors="pt"
)
inputs = tokenizer.build_inputs_for_generation(inputs, max_gen_length=64)

inputs = inputs.to(model.device)

output = model.generate(**inputs, max_new_tokens=64, eos_token_id=tokenizer.eop_token_id,do_sample = True)

tokenizer.decode(output[0].tolist(), skip_special_tokens=True)

In [ ]:
model.device

In [ ]:
import torch

model = model.eval()

def predict_test():
    predict = []
    with torch.inference_mode():
        for d in ds["test"]:
            inputs = tokenizer(
                "摘要生成: \n" + d["content"] + tokenizer.mask_token, return_tensors="pt"
            )
            inputs = tokenizer.build_inputs_for_generation(inputs, max_gen_length=64)
            inputs = inputs.to(model.device)
            output = model.generate(
                **inputs, max_new_tokens=64, eos_token_id=tokenizer.eop_token_id, do_sample=True
            )
            predict.append(tokenizer.decode(output[0].tolist().split("<|startofpiece|>")[1].replace("<endofpiece|>", "").strip(), skip_special_tokens=True))
            print("curID:", len(predict), "total:", len(ds["test"]))
    return predict

In [ ]:
results = predict_test()

In [ ]:
from rouge_chinese import Rouge

rouge = Rouge()

decode_preds = ["".join(p) for p in results]
decode_labels = ["".join(l) for l in ds["test"]["title"]]

scores = rouge.get_scores(decode_preds, decode_labels, avg=True)
{
    "rouge-1": scores["rouge-1"],
    "rouge-2": scores["rouge-2"],
    "rouge-l": scores["rouge-l"],
}